# Fake News Detector — LIAR dataset

In [ ]:
import os, zipfile, urllib.request
import pandas as pd

DATA_DIR = 'data'
URL = 'https://www.cs.ucsb.edu/~william/data/liar_dataset.zip'
ZIP_PATH = os.path.join(DATA_DIR, 'liar_dataset.zip')

os.makedirs(DATA_DIR, exist_ok=True)
if not os.path.exists(ZIP_PATH):
    print('Pobieram LIAR...')
    urllib.request.urlretrieve(URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)
    print('Gotowe.')
else:
    print('Plik już pobrany.')

print(os.listdir(DATA_DIR))

Pobieram LIAR...
Gotowe.
['valid.tsv', 'train.tsv', 'liar_dataset.zip', 'README', 'test.tsv']


Wypisuję kolumny z readme:

In [ ]:
COLUMNS = [
    'id', 'label', 'statement', 'subject', 'speaker', 'job_title',
    'state', 'party', 'barely_true_counts', 'false_counts',
    'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts',
    'context'
]

train = pd.read_csv(os.path.join(DATA_DIR, 'train.tsv'), sep='\t', header=None, names=COLUMNS)
valid = pd.read_csv(os.path.join(DATA_DIR, 'valid.tsv'), sep='\t', header=None, names=COLUMNS)
test  = pd.read_csv(os.path.join(DATA_DIR, 'test.tsv'),  sep='\t', header=None, names=COLUMNS)

print(f'train: {train.shape}, valid: {valid.shape}, test: {test.shape}')
print('Kolumny:', list(train.columns))
train.head(5)

train: (10240, 14), valid: (1284, 14), test: (1267, 14)
Kolumny: ['id', 'label', 'statement', 'subject', 'speaker', 'job_title', 'state', 'party', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context']


,id,label,statement,subject,speaker,job_title,state,party,barely_true_counts,false_counts,half_true_counts,mostly_true_counts,pants_on_fire_counts,context
0,2635.json,false,Says the Annies List political group supports ...,abortion,dwayne-bohac,State representative,Texas,republican,0.0,1.0,0.0,0.0,0.0,a mailer
1,10540.json,half-true,When did the decline of coal start? It started...,"energy,history,job-accomplishments",scott-surovell,State delegate,Virginia,democrat,0.0,0.0,1.0,1.0,0.0,a floor speech.
2,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama,President,Illinois,democrat,70.0,71.0,160.0,163.0,9.0,Denver
3,1123.json,false,Health care reform legislation is likely to ma...,health-care,blog-posting,NaN,NaN,none,7.0,19.0,3.0,5.0,44.0,a news release
4,9028.json,half-true,The economic turnaround started at the end of ...,"economy,jobs",charlie-crist,NaN,Florida,democrat,15.0,9.0,20.0,19.0,2.0,an interview on CNN


## Usuwam kolumny, które nie wnoszą sygnału do detekcji fake news

- **`id`** — identyfikator rekordu
- **`speaker`, `job_title`, `state`, `party`** — dane o mówcy (imie i nazwisko, praca, stan, partia pol.)

In [ ]:
cols_to_drop = ['id', 'speaker', 'job_title', 'state', 'party']

train = train.drop(columns=cols_to_drop)
valid = valid.drop(columns=cols_to_drop)
test  = test.drop(columns=cols_to_drop)

## Zmienna celu `y` = `label`

Kolumna `label` to nasza zmienna zależna (target). Sprawdzamy, jakie wartości może przyjmować.

In [ ]:
labels_train = sorted(train['label'].unique())

print('Unikalne etykiety:', labels_train)

Unikalne etykiety: ['barely-true', 'false', 'half-true', 'mostly-true', 'pants-fire', 'true']


## Zamiana etykiet tylko na true i false, sprawdzenie wielkości zbiorów

- **`false`** ← `pants-fire`, `false`, `barely-true`
- **`true`** ← `half-true`, `mostly-true`, `true`


In [ ]:
LABEL_MAP = {
    'pants-fire':  'false',
    'false':       'false',
    'barely-true': 'false',
    'half-true':   'true',
    'mostly-true': 'true',
    'true':        'true',
}

train['label'] = train['label'].map(LABEL_MAP)
valid['label'] = valid['label'].map(LABEL_MAP)
test['label']  = test['label'].map(LABEL_MAP)

print('Rozkład etykiet po binaryzacji:')
print('train:', train['label'].value_counts().to_dict())
print('valid:', valid['label'].value_counts().to_dict())
print('test :', test['label'].value_counts().to_dict())

Rozkład etykiet po binaryzacji:
train: {'true': 5752, 'false': 4488}
valid: {'true': 668, 'false': 616}
test : {'true': 714, 'false': 553}


## Mapowanie etykiet `'true'`/`'false'` → `1`/`0`

In [ ]:
LABEL_TO_INT = {'false': 0, 'true': 1}

train['label'] = train['label'].map(LABEL_TO_INT)
valid['label'] = valid['label'].map(LABEL_TO_INT)
test['label']  = test['label'].map(LABEL_TO_INT)

print(train['label'].value_counts().to_dict())

{1: 5752, 0: 4488}


## Preprocessing tekstu — wersja do porównania

Tworzę drugą wersję zbioru danych - z preprocessingiem (lowercase, usunięcie URL-i, interpunkcji, cyfr w słowach). Później chcę trenować RoBERTę na obu wersjach (surowej i przetworzonej) i porównać wyniki. ! Wazne ze interpunkcja wpływa na styl wypowiedzi, więc pytanie czy to nie obnizy jakosci wyniku ???

! todo: zobaczyc ilosc tokenow na wersji preprocessed, moze bedzie mozna wziac mniejsza ilosc niz 128

In [ ]:
import re
import string

def preprocess(text):
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\w*\d\w*', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_prep = train.copy()
valid_prep = valid.copy()
test_prep  = test.copy()

train_prep['statement'] = train_prep['statement'].apply(preprocess)
valid_prep['statement'] = valid_prep['statement'].apply(preprocess)
test_prep['statement']  = test_prep['statement'].apply(preprocess)

print('PRZED:', train['statement'].iloc[0])
print('PO:   ', train_prep['statement'].iloc[0])

PRZED: Says the Annies List political group supports third-trimester abortions on demand.
PO:    says the annies list political group supports thirdtrimester abortions on demand


In [ ]:
%pip install -q transformers torch

wzielam pre trained roberta-base; 2 labelki bo zmienilam y na 2 klasy:

In [ ]:
from transformers import RobertaForSequenceClassification

model_roberta = RobertaForSequenceClassification.from_pretrained('roberta-base',num_labels=2)
# 'roberta-base' : 12 layer, 768 hidden, 12 heads, 125M params RoBERTa using BERT-base architecture

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Do tokenizacji uzywam  tokenizatora AutoTokenizer (podobno szybszy niz RobertaTokenizer + mozna szybko zmienic na inny tokenizer)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Sprawdzamy ilości tokenów dla wypowiedzi:

In [ ]:
import numpy as np

encoded_train = [tokenizer.encode(s) for s in train['statement']]

lengths = [len(e) for e in encoded_train]
print(f'min={min(lengths)}, max={max(lengths)}')

print('pierwsza wypowiedz:')
print('Tekst :', train['statement'].iloc[0])
print('IDs   :', encoded_train[0])
print('Tokeny:', tokenizer.convert_ids_to_tokens(encoded_train[0]))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (911 > 512). Running this sequence through the model will result in indexing errors


min=4, max=911
pierwsza wypowiedz:
Tekst : Says the Annies List political group supports third-trimester abortions on demand.
IDs   : [0, 104, 4113, 5, 3921, 918, 9527, 559, 333, 4548, 371, 12, 4328, 38417, 17600, 15, 1077, 4, 2]
Tokeny: ['<s>', 'S', 'ays', 'Ġthe', 'ĠAnn', 'ies', 'ĠList', 'Ġpolitical', 'Ġgroup', 'Ġsupports', 'Ġthird', '-', 'tr', 'imester', 'Ġabortions', 'Ġon', 'Ġdemand', '.', '</s>']


In [ ]:
arr = np.array(lengths)
print(f'>128 tokenów: {(arr > 128).sum()} zdań')

>128 tokenów: 4 zdań


Widzimy, ze wiekszosc wypowiedzi ma ponizej 128 tokenow, wiec ustalam, ze na wejsciu bedzie taka wielkosc (roBERTa akceptuje od 1 do 512 tokenow). Dla kazdej wypowiedzi mamy 2 wektory (X) - wektor ztokenizowanych slow i wektor attention_mask - 1 oznacza ze jest slowo, 0 - padding.

In [ ]:
example = tokenizer(
    train['statement'].iloc[0],
    padding='max_length',
    max_length=128,
    truncation=True,
    return_tensors='pt'
)

print('Tekst:         ', train['statement'].iloc[0])
print('\ninput_ids:     ', example['input_ids'][0].tolist())
print('\nattention_mask:', example['attention_mask'][0].tolist())

Tekst:          Says the Annies List political group supports third-trimester abortions on demand.

input_ids:      [0, 104, 4113, 5, 3921, 918, 9527, 559, 333, 4548, 371, 12, 4328, 38417, 17600, 15, 1077, 4, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


tokenizacja train, test i validation set:

In [ ]:
import torch

def tokenize_set(df):
    return tokenizer(
        list(df['statement']),
        padding='max_length',
        max_length=128,
        truncation=True,
        return_tensors='pt'
    )

tokenize_trainset = tokenize_set(train)
tokenize_validset = tokenize_set(valid)
tokenize_testset  = tokenize_set(test)

print('train:', tokenize_trainset['input_ids'].shape)
print('valid:', tokenize_validset['input_ids'].shape)
print('test :', tokenize_testset['input_ids'].shape)

train: torch.Size([10240, 128])
valid: torch.Size([1284, 128])
test : torch.Size([1267, 128])


zamiana kolumn na torchowe tensory:

In [ ]:
y_train = torch.tensor(train['label'].tolist())
y_valid = torch.tensor(valid['label'].tolist())
y_test  = torch.tensor(test['label'].tolist())

print('y_train:', y_train.shape)
print('y_valid:', y_valid.shape)
print('y_test :', y_test.shape)

y_train: torch.Size([10240])
y_valid: torch.Size([1284])
y_test : torch.Size([1267])


opakowanie tokenow i etykiet w obiekt dataloader:

In [ ]:
class LiarDataset(torch.utils.data.Dataset):
    def __init__(self, enc, labels):
        self.enc = enc
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {
            'input_ids':      self.enc['input_ids'][i],
            'attention_mask': self.enc['attention_mask'][i],
            'labels':         self.labels[i]
        }

data loadery:

In [ ]:
train_loader = torch.utils.data.DataLoader(
    LiarDataset(tokenize_trainset, y_train),
    batch_size=16,
    shuffle=True #zeby pmodel nie pamietal kolejnosci
)

valid_loader = torch.utils.data.DataLoader(
    LiarDataset(tokenize_validset, y_valid),
    batch_size=16,
    shuffle=False
)

test_loader = torch.utils.data.DataLoader(
    LiarDataset(tokenize_testset, y_test),
    batch_size=16,
    shuffle=False
)

print('train batches:', len(train_loader))
print('valid batches:', len(valid_loader))
print('test batches: ', len(test_loader))

train batches: 640
valid batches: 81
test batches:  80


### Fine-tuning BERT vs RoBERTa

Trenujemy obie sieci na tym samym podziale danych (`train` → trening, `test` → ewaluacja) i porównujemy je metrykami precision / recall / F1. Prosta pętla treningowa w czystym PyTorchu.

Etykiety: `0 = false (fake)`, `1 = true (prawdziwa)`.

(Ewaluacji przed fine-tuningiem nie robimy — głowica klasyfikacyjna jest wtedy losowa, więc wynik byłby przypadkowy ~50% i nic nie wnosi.)

In [ ]:
%pip install -q scikit-learn

In [ ]:
import copy
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


def make_loader(df, tokenizer, shuffle):
    enc = tokenizer(list(df['statement']), padding='max_length',
                    truncation=True, max_length=128, return_tensors='pt')
    ds = LiarDataset(enc, torch.tensor(df['label'].tolist()))
    return torch.utils.data.DataLoader(ds, batch_size=16, shuffle=shuffle)


def predict(model, loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            logits = model(input_ids=ids, attention_mask=mask).logits
            preds += logits.argmax(1).cpu().tolist()
    return np.array(preds)

device: cuda


In [ ]:
def finetune(model_name, epochs=3, lr=2e-5):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

    train_loader = make_loader(train, tokenizer, shuffle=True)
    valid_loader = make_loader(valid, tokenizer, shuffle=False)
    test_loader = make_loader(test, tokenizer, shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    best_f1, best_state = -1.0, None
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            optimizer.zero_grad()
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            loss = model(input_ids=ids, attention_mask=mask, labels=labels).loss
            loss.backward()
            optimizer.step()

        valid_f1 = f1_score(valid['label'].values, predict(model, valid_loader), average='macro')
        print(f'{model_name} | epoch {epoch + 1}/{epochs} | valid_f1={valid_f1:.4f}')
        if valid_f1 > best_f1:
            best_f1, best_state = valid_f1, copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)  # bierzemy najlepszą epokę wg valid

    preds = predict(model, test_loader)
    trues = test['label'].values
    acc = accuracy_score(trues, preds)
    p, r, f1, _ = precision_recall_fscore_support(trues, preds, average='macro', zero_division=0)
    print(f'{model_name}: acc={acc:.4f} precision={p:.4f} recall={r:.4f} f1={f1:.4f} (best valid_f1={best_f1:.4f})')
    return {'name': model_name, 'accuracy': acc, 'precision': p, 'recall': r, 'f1': f1, 'preds': preds}

In [ ]:
res_bert = finetune('bert-base-uncased')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


bert-base-uncased | epoch 1/3 | valid_f1=0.5623
bert-base-uncased | epoch 2/3 | valid_f1=0.5991
bert-base-uncased | epoch 3/3 | valid_f1=0.6122
bert-base-uncased: acc=0.6275 precision=0.6202 recall=0.6063 f1=0.6039 (best valid_f1=0.6122)


In [ ]:
res_roberta = finetune('roberta-base')

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


roberta-base | epoch 1/3 | valid_f1=0.4617
roberta-base | epoch 2/3 | valid_f1=0.6245
roberta-base | epoch 3/3 | valid_f1=0.6555
roberta-base: acc=0.6401 precision=0.6333 recall=0.6220 f1=0.6215 (best valid_f1=0.6555)


In [ ]:
results = [res_bert, res_roberta]

ranking = (pd.DataFrame(results)[['name', 'accuracy', 'precision', 'recall', 'f1']]
           .sort_values('f1', ascending=False)
           .round(4)
           .reset_index(drop=True))

ranking

,name,accuracy,precision,recall,f1
0,roberta-base,0.6401,0.6333,0.6220,0.6215
1,bert-base-uncased,0.6275,0.6202,0.6063,0.6039


## Zero-shot i Few-shot przez OpenRouter

Testujemy 4 modele na tym samym podzbiorze testowym:

- `openai/gpt-oss-20b:free`
- `google/gemma-4-31b-it:free`
- `cognitivecomputations/dolphin-mistral-24b-venice-edition:free`
- `meta-llama/llama-3.3-70b-instruct:free`

Dla każdego modelu uruchamiamy:

- **zero-shot**: tylko instrukcja + zdanie,
- **few-shot**: instrukcja + kilka oznaczonych przykładów + zdanie.

Dla API LLM pełny zbior testowy (`1267` przykładów) będzie bardzo wolny. Domyślnie testujemy na mniejszej próbce (`MAX_TEST_SAMPLES`).

In [ ]:
%pip install -q requests tqdm

In [ ]:
import os
import re
import time
import requests
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

OPENROUTER_API_KEY = "REDACTED_API_KEY"
if not OPENROUTER_API_KEY:
    raise ValueError('Brak OPENROUTER_API_KEY w zmiennych środowiskowych.')

OPENROUTER_URL = 'https://openrouter.ai/api/v1/chat/completions'

MODELS = [
    'openai/gpt-oss-20b:free',
    'google/gemma-4-31b-it:free',
    'cognitivecomputations/dolphin-mistral-24b-venice-edition:free',
    'meta-llama/llama-3.3-70b-instruct:free',
]

MAX_TEST_SAMPLES = 120   # zwieksz, jesli chcesz pelniejsza ocene
TEMPERATURE = 0
MAX_TOKENS = 8
SLEEP_BETWEEN_CALLS = 0.15

LABEL_INT_TO_STR = {0: 'fake', 1: 'true'}
LABEL_STR_TO_INT = {'fake': 0, 'true': 1}

SYSTEM_PROMPT = (
    'You are a strict binary classifier for misinformation detection. '
    'Return ONLY one token: fake or true.'
)

# Proste, zbalansowane przyklady do few-shot (4 sztuki)
few_shot_examples = []
for label_int in [0, 1, 0, 1]:
    row = train[train['label'] == label_int].sample(1, random_state=42 + len(few_shot_examples)).iloc[0]
    few_shot_examples.append({'statement': row['statement'], 'label': LABEL_INT_TO_STR[label_int]})

few_shot_block = '\n'.join([
    f"statement: {e['statement']}\nlabel: {e['label']}" for e in few_shot_examples
])

print('Few-shot examples prepared:', len(few_shot_examples))

Few-shot examples prepared: 4


In [ ]:
def build_user_prompt(statement, mode='zero'):
    if mode == 'zero':
        return (
            'Task: classify if the statement is fake or true.\n'
            'Return only: fake OR true.\n\n'
            f'statement: {statement}'
        )

    return (
        'Task: classify if the statement is fake or true.\n'
        'Use examples below.\n'
        'Return only: fake OR true.\n\n'
        f'{few_shot_block}\n\n'
        f'statement: {statement}\n'
        'label:'
    )


def parse_label(text):
    t = text.strip().lower()
    # tolerancja na dodatkowy tekst typu "label: fake"
    m = re.search(r'\b(fake|true)\b', t)
    if not m:
        return None
    return m.group(1)


def call_openrouter(model, user_prompt, retries=3):
    headers = {
        'Authorization': f'Bearer {OPENROUTER_API_KEY}',
        'Content-Type': 'application/json',
    }
    payload = {
        'model': model,
        'temperature': TEMPERATURE,
        'max_tokens': MAX_TOKENS,
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_prompt},
        ],
    }

    for attempt in range(retries):
        try:
            resp = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=45)
            if resp.status_code == 200:
                content = resp.json()['choices'][0]['message']['content']
                return content

            # przy limicie/rate-limit robimy lekki backoff
            if resp.status_code in [429, 500, 502, 503, 504] and attempt < retries - 1:
                time.sleep(1.5 * (attempt + 1))
                continue

            return f'ERROR_STATUS_{resp.status_code}'

        except requests.RequestException:
            if attempt < retries - 1:
                time.sleep(1.5 * (attempt + 1))
                continue
            return 'ERROR_REQUEST'


def evaluate_model_llm(model, mode='zero', n_samples=MAX_TEST_SAMPLES):
    df = test.sample(n=min(n_samples, len(test)), random_state=42).reset_index(drop=True)

    y_true, y_pred = [], []
    invalid = 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f'{model} | {mode}'):
        prompt = build_user_prompt(row['statement'], mode=mode)
        raw = call_openrouter(model, prompt)
        lbl = parse_label(raw)

        if lbl is None:
            invalid += 1
            # fallback: traktuj jako fake, żeby zachować długość wektorów
            lbl = 'fake'

        y_true.append(row['label'])
        y_pred.append(LABEL_STR_TO_INT[lbl])

        time.sleep(SLEEP_BETWEEN_CALLS)

    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)

    return {
        'model': model,
        'mode': mode,
        'samples': len(df),
        'invalid_outputs': invalid,
        'accuracy': acc,
        'precision': p,
        'recall': r,
        'f1': f1,
    }

In [ ]:
llm_results = []

for model in MODELS:
    llm_results.append(evaluate_model_llm(model, mode='zero', n_samples=MAX_TEST_SAMPLES))
    llm_results.append(evaluate_model_llm(model, mode='few', n_samples=MAX_TEST_SAMPLES))

llm_results_df = pd.DataFrame(llm_results)
llm_results_df

openai/gpt-oss-20b:free | zero:   0%|          | 0/120 [00:00<?, ?it/s]

openai/gpt-oss-20b:free | few:   0%|          | 0/120 [00:00<?, ?it/s]

google/gemma-4-31b-it:free | zero:   0%|          | 0/120 [00:00<?, ?it/s]

In [ ]:
ranking_llm = (llm_results_df
               .sort_values(['f1', 'accuracy'], ascending=False)
               .reset_index(drop=True)
               .round(4))

print('Ranking (LLM zero-shot / few-shot):')
ranking_llm